# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR^2 human mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset metadata is provided by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(url)

# Print basic dataset info
md = dataset.metadata
print(f"{md.name}: {md.description}\n")
print(f"Published: {md.datePublished}")
print(f"Version: {md.version}")
print(f"Identifier: {md.identifier}")

## 2. Data Overview

Below, we list available record sets and their respective fields. All references are by entity `@id`.

In [ ]:
# List available record sets and their fields by @id

from pprint import pprint

print("RecordSets available in the dataset:\n")
record_sets = dataset.record_sets
record_set_ids = []

for rs in record_sets:
    print(f"@id: {rs['@id']}")
    print(f"  name: {rs.get('name', 'N/A')}")
    field_ids = [f['@id'] for f in rs.get('field', []) if isinstance(f, dict) and '@id' in f]
    if field_ids:
        print(f"  field @ids:")
        for fid in field_ids:
            print(f"    - {fid}")
    else:
        print("  (No fields found)")
    print()
    record_set_ids.append(rs['@id'])

# Choose the first RecordSet for demonstration
if record_set_ids:
    main_record_set_id = record_set_ids[0]
else:
    print("No record sets found in this dataset.")

print(f"Chosen record_set @id for further analysis: {main_record_set_id}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame. We use the record set and field `@id`s collected above.

In [ ]:
# Extract records from each record set and convert to DataFrame

dataframes = {}

for rs_id in record_set_ids:
    try:
        records_iter = dataset.records(record_set=rs_id)
        records_list = list(records_iter)
        if len(records_list) > 0:
            df = pd.DataFrame(records_list)
            dataframes[rs_id] = df
            print(f"record_set {rs_id}: Loaded {df.shape[0]} rows, {df.shape[1]} columns.")
        else:
            print(f"record_set {rs_id}: (No records returned)")
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

# Display columns in the main record set
if main_record_set_id in dataframes:
    print(f"\nColumns in record_set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print(f"No data available for {main_record_set_id}.")

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field (by `@id`) for basic filtering and normalization. We also show grouping by a categorical field (when available).

In [ ]:
# You may choose these according to the printed field @ids and dataframe columns above.

record_set_id = main_record_set_id

# Example: Try to find a plausible numeric field (e.g., 'coverage', 'mw', 'abundance_sample_1', etc.)
# For demonstration, we'll use the first numeric-looking column found.

import numpy as np

df = dataframes[record_set_id]

numeric_candidate = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]) or np.issubdtype(df[col].dtype, np.number):
        numeric_candidate = col
        break

# Fallback: If none found, try to coerce one.
if numeric_candidate is None:
    for col in df.columns:
        # Try casting to float
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if df[col].notnull().sum() > 0:
                numeric_candidate = col
                break
        except Exception:
            continue

if numeric_candidate is None:
    raise RuntimeError("No numeric column found for EDA.")
else:
    print(f"Numeric field chosen for filtering: {numeric_candidate}")

# Simple threshold filtering
threshold = df[numeric_candidate].mean() if df[numeric_candidate].notnull().sum() > 0 else 0
filtered_df = df[df[numeric_candidate] > threshold]
print(f"Filtered records where {numeric_candidate} > {threshold:.2f} (mean): {filtered_df.shape[0]} rows\n")

# Normalization
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_candidate}_normalized"] = (
    (filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()) / filtered_df[numeric_candidate].std()
)
print(filtered_df[[numeric_candidate, f"{numeric_candidate}_normalized"]].head())

# Try grouping by the first non-numeric field
group_field = None
for col in df.columns:
    if not pd.api.types.is_numeric_dtype(df[col]) and col != numeric_candidate:
        group_field = col
        break
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_candidate].mean().reset_index()
    print(f"\nGrouping filtered records by {group_field} (mean of {numeric_candidate}):")
    print(grouped_df.head())

## 5. Visualization

We can plot the distribution of the numeric field and, if applicable, boxplot by a group. Install seaborn if not available.

In [ ]:
# Plotting distributions with seaborn and matplotlib
!pip install seaborn --quiet
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(7,4))
sns.histplot(filtered_df[numeric_candidate].dropna(), kde=True, ax=ax)
ax.set_title(f'Distribution of {numeric_candidate}')
plt.show()

if group_field:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field, y=numeric_candidate, data=filtered_df)
    plt.xticks(rotation=90)
    plt.title(f'{numeric_candidate} by {group_field}')
    plt.show()

## 6. Conclusion

We have loaded the FAIR^2 mass spectrometry dataset using `mlcroissant`, identified available record sets and fields by `@id`, performed initial filtering and normalization on one numeric field, and visualized its distribution and group-wise statistics. For further analysis, you may select specific fields by their `@id`, apply domain-specific filters, or extend the visualization section as needed.